In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F
block_size = 8
batch_size = 4
max_iters = 1000
learning_rate = 3e-4
eval_iters = 250
#dropout = 0.2
device = "mps" if torch.backends.mps.is_available() else "cpu"

In [10]:
with open ('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '&', '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”']


In [11]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([41, 55, 52,  1, 44, 62, 61, 51, 52, 65, 53, 68, 59,  1, 44, 56, 73, 48,
        65, 51,  1, 62, 53,  1, 36, 73,  0,  0, 49, 72,  1, 33,  8,  1, 27, 65,
        48, 61, 58,  1, 23, 48, 68, 60,  0,  0,  0, 41, 55, 56, 66,  1, 49, 62,
        62, 58,  1, 56, 66,  1, 51, 52, 51, 56, 50, 48, 67, 52, 51,  1, 67, 62,
         1, 60, 72,  1, 54, 62, 62, 51,  1, 53, 65, 56, 52, 61, 51,  1,  3,  1,
        50, 62, 60, 65, 48, 51, 52,  0, 34, 72])


In [12]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('x:', x.shape)
print(x)
print('y:', y.shape)
print(y)

x: torch.Size([4, 8])
tensor([[56, 60,  1, 48, 54, 48, 56, 61],
        [ 1, 56, 61, 51, 52, 52, 51,  1],
        [60, 72,  1, 62, 65,  1, 48,  1],
        [54, 62,  6,  1, 53, 62, 65,  1]], device='mps:0')
y: torch.Size([4, 8])
tensor([[60,  1, 48, 54, 48, 56, 61,  8],
        [56, 61, 51, 52, 52, 51,  1, 49],
        [72,  1, 62, 65,  1, 48,  1, 51],
        [62,  6,  1, 53, 62, 65,  1, 30]], device='mps:0')


In [13]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [14]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets =None):
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)    


.XrrnCDw3cri6e,N?lK‘Y;r—I-&zXls9s7hv4)I.8
7Hijh2-3mnd9 iSciAo;8I6yDnl4&f0-Gv-T)—AdI7hcX
BQJFJ&?a&YQ.5s)I—VeLgob
EqfyIanSu;rExmSDW8gwN8J&QJ4G8&5mHBS 9“f7fKh3F21SbNY3n,9kQ5MFk&25GYlx:S?2c(
DfL poqUR
E&Q:5250wX,.l9kSePru“0aGnjhvS0s)GPZk:g6qH“t,64veFeOWfQ”30’D9sxEwhW(DyB‘5FtnKV5v3gMuAk:HnkJ&MDm2CCvmS)A,—N:.lx‘EnXghpom4EghrAoq8gwby0vl’CP4N(PlLsvkxMOyOp2’Xa8nkQ.u(I50HVbLQ9in&PzkB8W?OSo8stpk( TWKR6G1dNc‘—pzjhSpvPr—Gv—
mmH66.lE07sXydOR0wcMgi—JiRbqH(4:cvMuv9x)knmn4K1Soms”lW8nfyD-FUM6pJPrjupkQ;ZsV?6mSfQ;F


In [17]:
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss {losses['train']:.3f}, val loss {losses['val']:.3f}")

    xb, yb = get_batch('train')

    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss 4.640, val loss 4.620
step: 250, train loss 4.564, val loss 4.540
step: 500, train loss 4.515, val loss 4.514
step: 750, train loss 4.441, val loss 4.426
4.618430137634277


In [16]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens= 5000)[0].tolist())
print(generated_chars)


HwODG620,m1OPfCmy!Yghe-MnZv9F(hMurf,Pf,J4—41tp Y)I.8J”ZsKpds fryG!lG5MCl?pU5FRZl40cSow9BS—g;SC’L2 b“pk:,yLQ w(V—b)x5J4-”1“—“7My0 ,u”Unt IY;DTZEdz!e2qxIXCPoGvLQ(OPTg;SpkNCd.’Y3s yvkghZs8G;n!f6qr—Rl7x5LNP-&iDxkBJp8”‘
8lA5‘Y&A7’m4hMMd”s8—:G6P6U6”KDGzs
50MZl3Gl‘ORa62’Y5AcKcDE.c:An4EnUzyN:N—YDv!tkUIaOt..uGlg9w’o8SII?aA0RYQ,u,.di1mC’”D1M&41OV&0ptJKA,mZz5xWvcpuMNi
cf7;Rp4vu‘SYbxCF‘8Ji0mF3up4Kv23YBx5’)s —GD9Dz8r9(SocKp—OCV—
F(3pF.zAaf6—Gcf62k3c“K’8SqHkqnsB8:DsW-ENFR‘9u.1v—
82FXWU8”JVS“(o;”MI3dRmjlhMNJ“yDSDR“FU5y1OEwrA7NXXz7L.X)(GB:1Szh9&2)LpdZ1t11&-)Z”nAxm,DGB‘LIVZXhvT)Rmy0pHwq?; pz;S“5 qDT:p2srMZi0pVda,r4X(K06.z7uOYEO”M&cT‘,evUWqW1 liDm1n!uMR6Ttk,ZkMB3FaPorUT“p.)Sq3Ny0“KGlPTKb;ak)I-1ORt XG’Ym’8SoLpfz8JUtW:liYQrkQGLIyLQHbwaf64 “qV“H(u’Q3xBbg?S—l-);VJOS
N:!ZyBCdQ5M”5
bxB’,GpLwSDn&vZs8?50lxs,6Lg—X2?q,.uH(cWlYNONupV641e“tQ9
7sSqA’ lw(OSF(hZRZ“MR6pq.9ndxriN:H(h—
ktpMJLQJeFcM7GL8Mej”
Nec1dvZsgZ4Usja8vy?54fBWs bn“LqDB:cC’Cds ’gNO‘
mnUyN3‘Z“7;XmndfQ qGD—
5HXs.KE3e7zQKm‘sS( q,6RC’io xr’It.,6vEIOVxkxd